# Fase 2.2 — EDA bivariado: relación de cada variable con el incumplimiento

**Tesis v2 · Eider Johan Díaz Berrío · MCDA EAFIT**

Aquí miro por primera vez la relación entre los predictores candidatos y el evento 90+/12m. Fijo tres reglas antes de empezar:

- **Solo cohortes anteriores a 2025** (494 créditos, 16 eventos). Así el *backtest* temporal secundario (2025+) no se contamina con decisiones tomadas aquí. Ya fue consultado en la v1, pero no quiero sumar más miradas.
- **Todo intervalo es por *bootstrap* de clientes** (500 réplicas sobre grupos de cliente), no de créditos, porque los créditos del mismo cliente no son independientes.
- **El signo esperado se fijó antes** (diccionario de la Fase 1). Aquí solo lo contrasto.

Con 16 eventos, cada estadística bivariada es muy ruidosa. El objetivo no es elegir variables por su IV (eso sería sobreajustar), sino detectar señales consistentes con el dominio, variables sin ninguna señal y relaciones con el signo contrario al esperado que merezcan revisión.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from tesis_v2 import config, estadistica as est, figuras, io_datos

warnings.filterwarnings("ignore")
pd.set_option("display.width", 170, "display.max_columns", 30, "display.max_rows", 60)
figuras.estilo()
B = 500

ds = io_datos.leer_parquet(config.PRIVADO / "dataset_analitico_v2.parquet")
dic = pd.read_csv(config.RESULTADOS / "fase1" / "diccionario_v2.csv").set_index("variable")
lib = ds[ds["producto"] == "libranza"]
dev = lib[lib["elegible"] & (lib["conjunto_temporal"] == "<2025")].reset_index(drop=True)
dev["y"] = dev["y_90_12m"].astype(int)
print(f"Desarrollo: {len(dev)} créditos, {dev['y'].sum()} eventos, {dev['grupo_cliente'].nunique()} grupos de cliente")

Desarrollo: 494 créditos, 16 eventos, 271 grupos de cliente


## 1. Tasa de evento por categoría

Para las variables categóricas muestro la tasa con su IC de Wilson. Con tan pocos eventos, un IC que va de 0 % a 10 % es la norma, no la excepción.

In [2]:
CATS = ["anio_cohorte", "recaudo", "tipo_solicitud", "es_recurrente", "cierre_previo", "tiene_aval", "cobra_seguro",
        "cobra_estudio", "cobra_4x1000"]
tasas = pd.concat([est.tabla_tasa(dev.assign(categoria=dev[v].astype(str)), "categoria", "y").assign(variable=v)
                   for v in CATS], ignore_index=True)
io_datos.guardar_resultado(tasas, "fase2", "tasa_evento_por_categoria_desarrollo")
tasas[["variable", "categoria", "creditos", "eventos", "tasa_%", "ic95_inf_%", "ic95_sup_%"]]

,variable,categoria,creditos,eventos,tasa_%,ic95_inf_%,ic95_sup_%
0,anio_cohorte,2022,47,4,8.51,3.36,19.93
1,anio_cohorte,2023,197,5,2.54,1.09,5.80
2,anio_cohorte,2024,250,7,2.80,1.36,5.67
3,recaudo,empresarial,294,10,3.40,1.86,6.15
4,recaudo,mensual,178,6,3.37,1.55,7.16
5,recaudo,quincenal,22,0,0.00,0.00,14.87
6,tipo_solicitud,credito,319,10,3.13,1.71,5.67
7,tipo_solicitud,refinanciacion_anterior,6,0,0.00,0.00,39.03
8,tipo_solicitud,sin_solicitud,169,6,3.55,1.64,7.53
9,es_recurrente,0,259,9,3.47,1.84,6.47


In [3]:
fig, axes = plt.subplots(3, 3, figsize=(10, 7.4), sharex=True)
for ax, v in zip(axes.flat, CATS):
    t = tasas[tasas["variable"] == v].reset_index(drop=True)
    y = np.arange(len(t))
    ax.errorbar(t["tasa_%"], y, xerr=[t["tasa_%"] - t["ic95_inf_%"], t["ic95_sup_%"] - t["tasa_%"]],
                fmt="o", color=figuras.AZUL, ecolor=figuras.GRIS, capsize=2, ms=4)
    ax.axvline(100 * dev["y"].mean(), color=figuras.NARANJA, ls="--", lw=0.8)
    ax.set_yticks(y, [f"{c} (n={n})" for c, n in zip(t["categoria"], t["creditos"])], fontsize=7)
    ax.set_title(v, fontsize=8.5)
    ax.invert_yaxis()
for ax in axes[-1]:
    ax.set_xlabel("Tasa 90+/12m (%) e IC 95 %", fontsize=8)
fig.suptitle("Tasa de incumplimiento por categoría — desarrollo (< 2025); línea = tasa global", fontweight="bold", fontsize=10)
fig.tight_layout()
figuras.guardar(fig, "fase2", "E06", "tasa_por_categoria",
                "Casi todos los IC se superponen con la tasa global: con 16 eventos ninguna categoría se separa con claridad.")

**Lectura.** Casi todos los intervalos se superponen con la tasa global (3,2 %). La única diferencia que se insinúa es la de la cohorte 2022 (4 eventos en 47 créditos). El recaudo, que en el notebook 05 resultó ser un indicador de época, no se separa. Lo compruebo dentro de 2023, el único año en que conviven las tres modalidades:

In [4]:
c23 = dev[dev["anio_cohorte"] == 2023]
est.tabla_tasa(c23, "recaudo", "y")

,recaudo,creditos,eventos,tasa_%,ic95_inf_%,ic95_sup_%
0,empresarial,44,3,6.82,2.35,18.23
1,mensual,131,2,1.53,0.42,5.40
2,quincenal,22,0,0.00,0.00,14.87


Dentro de 2023 la relación incluso **se invierte**: 3 eventos en 44 créditos empresariales (6,8 %) frente a 2 en 131 mensuales (1,5 %), con intervalos que se superponen. La señal "recaudo directo = más riesgo" de la v1 venía de mezclar productos y cohortes. Esto refuerza la recomendación de no tratarlo como rasgo del cliente.

## 2. WoE e IV con intervalos por *bootstrap* de clientes

Discretizo cada variable numérica en cuatro intervalos por cuantiles. Con 16 eventos, más intervalos dejarían celdas vacías. Los faltantes forman su propia categoría: en la mora previa y la antigüedad, "Faltante" significa "sin crédito previo". El WoE usa suavizado de Laplace (0,5) para que una celda sin eventos no produzca un valor infinito. Solo analizo variables con al menos 50 % de datos en desarrollo; el buró y la PILA no existen en estas cohortes.

Una advertencia de método: el IV está **sesgado hacia arriba con pocos eventos**. Aun una variable sin relación alguna con el evento tiene un IV esperado positivo. Para tener una referencia, calculo también el IV de una variable de ruido puro con la misma estructura (cuatro intervalos), permutando el evento.

In [5]:
NUM = ["monto", "plazo_meses", "cuota_mensual", "tasa_mensual", "edad_t0", "n_creditos_previos", "n_activos_t0",
       "peor_mora_previa", "meses_desde_primer_credito", "emp_creditos_previos"]
BIN = ["es_recurrente", "es_refinanciacion", "tiene_aval", "cobra_seguro", "cobra_estudio", "cobra_4x1000"]
CAT = ["recaudo", "tipo_solicitud", "cierre_previo"]
VARS = [v for v in NUM + BIN + CAT if dev[v].notna().mean() >= 0.5 or v in {"peor_mora_previa", "meses_desde_primer_credito"}]


def codificar(df, v, cortes=None):
    if v in NUM:
        return est.discretizar(df[v].astype(float), 4, cortes)
    return df[v].astype(str), None


filas, perfiles = [], {}
for v in VARS:
    cat, cortes = codificar(dev, v)
    tabla, iv = est.woe_iv(cat, dev["y"])
    perfiles[v] = tabla

    def iv_boot(sub, v=v, cortes=cortes):
        return est.woe_iv(codificar(sub, v, cortes)[0], sub["y"])[1]

    bs = est.bootstrap_por_cliente(dev, "grupo_cliente", iv_boot, B=B)
    filas.append({"variable": v, "iv": iv, "iv_ic95_inf": np.nanpercentile(bs, 2.5), "iv_ic95_sup": np.nanpercentile(bs, 97.5),
                  "categorias": len(tabla)})
iv_tab = pd.DataFrame(filas)

rng = np.random.default_rng(config.SEMILLA)
ruido = [est.woe_iv(pd.Series(rng.integers(0, 4, len(dev))).astype(str), pd.Series(rng.permutation(dev["y"].values)))[1]
         for _ in range(1000)]
iv_ruido_p95 = float(np.percentile(ruido, 95))
print(f"IV de una variable de ruido con 4 categorías: mediana {np.median(ruido):.3f}, percentil 95 {iv_ruido_p95:.3f}")

IV de una variable de ruido con 4 categorías: mediana 0.126, percentil 95 0.446


## 3. AUC univariada y contraste con el signo esperado

Para las variables numéricas y binarias calculo el AUC de la variable sola, con IC por *bootstrap* de clientes. Omito las binarias casi constantes en desarrollo (menos de 10 créditos en la categoría minoritaria), porque su AUC es degenerado. Un AUC mayor que 0,5 significa que valores más altos se asocian con más incumplimiento (signo "+"). En la mora previa y la antigüedad, el faltante se codifica como 0, porque sin crédito previo no hubo mora previa. Las categóricas solo tienen IV: su AUC "en muestra" sería optimista por construcción.

In [6]:
def preparar(df, v):
    x = df[v].astype(float)
    return x.fillna(0) if v in {"peor_mora_previa", "meses_desde_primer_credito"} else x


aucs = []
minoritaria = {v: dev[v].value_counts().min() if dev[v].nunique() == 2 else 999 for v in BIN}
for v in [v for v in VARS if v in NUM + BIN and minoritaria.get(v, 999) >= 10 and dev[v].nunique() > 1]:
    a = est.auc_orientado(preparar(dev, v), dev["y"])
    bs = est.bootstrap_por_cliente(dev, "grupo_cliente", lambda s, v=v: est.auc_orientado(preparar(s, v), s["y"]), B=B)
    lo, hi = np.nanpercentile(bs, [2.5, 97.5])
    obs = "+" if a > 0.5 else "−"
    esp = dic.loc[v, "signo_esperado"] if v in dic.index else "?"
    aucs.append({"variable": v, "auc": a, "auc_ic95_inf": lo, "auc_ic95_sup": hi, "signo_observado": obs,
                 "signo_esperado": esp, "ic_excluye_05": (lo > 0.5) or (hi < 0.5),
                 "coincide_signo": "—" if esp not in {"+", "−"} else ("sí" if esp == obs else "no")})
auc_tab = pd.DataFrame(aucs)

biv = iv_tab.merge(auc_tab, on="variable", how="left")
biv["iv_supera_ruido_p95"] = biv["iv"] > iv_ruido_p95
biv = biv.sort_values("iv", ascending=False)
io_datos.guardar_resultado(biv.round(4), "fase2", "woe_iv_auc_desarrollo")
biv.round(3)

,variable,iv,iv_ic95_inf,iv_ic95_sup,categorias,auc,auc_ic95_inf,auc_ic95_sup,signo_observado,signo_esperado,ic_excluye_05,coincide_signo,iv_supera_ruido_p95
0,monto,0.626,0.449,1.130,4,0.666,0.546,0.767,+,?,True,—,True
2,cuota_mensual,0.451,0.128,1.335,5,0.678,0.536,0.794,+,?,True,—,True
9,emp_creditos_previos,0.247,0.047,1.213,4,0.306,0.176,0.448,−,−,True,sí,False
18,cierre_previo,0.213,0.102,0.790,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
15,cobra_4x1000,0.121,0.001,0.712,2,0.591,0.465,0.702,+,?,False,—,False
8,meses_desde_primer_credito,0.066,0.040,0.630,5,0.457,0.339,0.567,−,−,False,sí,False
1,plazo_meses,0.064,0.021,0.764,4,0.574,0.426,0.720,+,+,False,sí,False
4,edad_t0,0.063,0.017,0.730,5,0.426,0.279,0.559,−,−,False,sí,False
3,tasa_mensual,0.055,0.025,0.781,4,0.532,0.369,0.672,+,+,False,sí,False
7,peor_mora_previa,0.032,0.015,0.511,4,0.436,0.342,0.534,−,+,False,no,False


In [7]:
fig, ax = plt.subplots(figsize=(7.2, 0.32 * len(biv) + 1.4))
b2 = biv.iloc[::-1]
y = np.arange(len(b2))
col = [figuras.AZUL if s else figuras.GRIS for s in b2["iv_supera_ruido_p95"]]
ax.hlines(y, b2["iv_ic95_inf"], b2["iv_ic95_sup"], color=col, lw=1.6)
ax.scatter(b2["iv"], y, color=col, zorder=3, s=22)
ax.axvline(iv_ruido_p95, color=figuras.NARANJA, ls="--", lw=0.9)
ax.text(iv_ruido_p95, len(b2) - 0.4, " p95 del ruido", color=figuras.NARANJA, fontsize=7.5)
ax.set_yticks(y, b2["variable"], fontsize=8)
ax.set_xlabel("Information Value (IC 95 % bootstrap por cliente)")
ax.set_title("IV por variable — desarrollo (< 2025), evento 90+/12m")
figuras.guardar(fig, "fase2", "E07", "iv_con_ic",
                "Solo unas pocas variables superan el IV esperado por puro ruido, y todos los IC son muy anchos.")

**Lectura.** Los intervalos de IV son muy anchos, y la referencia de ruido (línea naranja) es alta: con 16 eventos, una variable aleatoria de cuatro categorías tiene un IV mediano de 0,13 y llega a 0,45 en el percentil 95. En la práctica de *scorecards* eso se llamaría señal "media" o "fuerte". Solo el **monto** supera esa referencia, con su variante redundante, la cuota. Su AUC univariado (0,67) tiene un IC que excluye 0,5: créditos más grandes, más incumplimiento.

El contraste con los signos esperados deja tres cosas:

- **Coinciden** la antigüedad del empleador (`emp_creditos_previos`, AUC 0,31: más trayectoria del convenio, menos incumplimiento; IC que excluye 0,5), la edad, el plazo, la antigüedad del cliente y el número de créditos previos. Es validez de constructo, aunque casi todos los IC incluyen 0,5.
- **No coinciden** la peor mora previa y los créditos activos en t0: el signo observado es negativo, contra el esperado positivo. Los IC incluyen 0,5, así que no hay evidencia de un signo invertido, solo de ausencia de señal con estos datos. No los "corrijo" con los datos: se revisan en la Fase 3 por dominio.
- `cobra_seguro` y `cobra_estudio` son **constantes** antes de 2025: son parámetros de producto que la fintech empezó a cobrar después. En este periodo no pueden aprender nada.

In [8]:
top = [v for v in biv["variable"] if v in NUM][:6]
fig, axes = plt.subplots(2, 3, figsize=(10, 5.6))
for ax, v in zip(axes.flat, top):
    t = perfiles[v].reset_index()
    t.columns = ["intervalo"] + list(t.columns[1:])
    ax.bar(range(len(t)), t["tasa_%"], color=figuras.AZUL, width=0.6)
    for i, (r, n, e) in enumerate(zip(t["tasa_%"], t["creditos"], t["eventos"])):
        ax.text(i, r + 0.3, f"{e}/{n}", ha="center", fontsize=7)
    ax.set_xticks(range(len(t)), t["intervalo"].astype(str), rotation=35, ha="right", fontsize=6.5)
    ax.set_title(v, fontsize=8.5)
    ax.set_ylabel("Tasa 90+/12m (%)", fontsize=7.5)
    ax.grid(axis="x", visible=False)
fig.suptitle("Perfil de la tasa de evento por intervalo (eventos/créditos) — variables numéricas con mayor IV",
             fontweight="bold", fontsize=10)
fig.tight_layout()
figuras.guardar(fig, "fase2", "E08", "perfiles_woe",
                "Los perfiles de las variables con más IV se apoyan en 2 a 6 eventos por intervalo: son señales, no relaciones estimadas.")

## 4. Robustez: ¿se repiten las señales con más eventos?

Si una señal es real, debería aparecer también con los objetivos auxiliares, que tienen más eventos en las mismas cohortes: 60+/12m (20 eventos) y 30+/6m (44 eventos). Repito el IV y comparo el ordenamiento con la correlación de Spearman.

In [9]:
def poblacion(df, ev):
    mad = "madura_6m" if ev == "y_30_6m" else "madura_12m"
    return df[df[mad] & (df["cierre_v2"] != "INDETERMINADO") & (df["anio_cohorte"] != 2021)
              & (df["conjunto_temporal"] == "<2025")].reset_index(drop=True)


rob = {"y_90_12m": biv.set_index("variable")["iv"]}
for ev in ["y_60_12m", "y_30_6m"]:
    p = poblacion(lib, ev)
    rob[ev] = pd.Series({v: est.woe_iv(codificar(p, v)[0], p[ev].astype(int))[1] for v in VARS})
    print(f"{ev}: {len(p)} créditos, {int(p[ev].sum())} eventos")
rob = pd.DataFrame(rob).sort_values("y_90_12m", ascending=False)
io_datos.guardar_resultado(rob.round(4).reset_index(names="variable"), "fase2", "iv_robustez_objetivos")
print("\nCorrelación de Spearman entre los IV:")
print(rob.corr(method="spearman").round(2).to_string())
rob.round(3)

y_60_12m: 494 créditos, 20 eventos
y_30_6m: 494 créditos, 44 eventos

Correlación de Spearman entre los IV:
          y_90_12m  y_60_12m  y_30_6m
y_90_12m      1.00      0.86     0.75
y_60_12m      0.86      1.00     0.73
y_30_6m       0.75      0.73     1.00


,y_90_12m,y_60_12m,y_30_6m
monto,0.626,0.183,0.042
cuota_mensual,0.451,0.189,0.115
emp_creditos_previos,0.247,0.257,0.264
cierre_previo,0.213,0.135,0.137
cobra_4x1000,0.121,0.243,0.015
meses_desde_primer_credito,0.066,0.112,0.109
plazo_meses,0.064,0.016,0.101
edad_t0,0.063,0.043,0.077
tasa_mensual,0.055,0.152,0.062
peor_mora_previa,0.032,0.063,0.090


**Lectura.** Los ordenamientos son razonablemente estables (Spearman de 0,73 a 0,86), pero **la variable más fuerte con 90+/12m no se sostiene**. El IV del monto cae de 0,63 a 0,18 con 60+/12m y a 0,04 con 30+/6m. Lo más probable es que refleje unos pocos eventos de créditos grandes, no una relación general. La señal más consistente es la antigüedad del empleador, con un IV de alrededor de 0,25 en las tres definiciones. También el cierre del crédito previo se repite (0,13 a 0,21). Una advertencia: `emp_creditos_previos` crece con el calendario (es un conteo acumulado), así que parte de su señal puede ser de época, como en el recaudo. En la Fase 3 la evaluaré también dentro de cohorte.

## 5. Del EDA a decisiones de la Fase 3

Traduzco lo anterior en una propuesta explícita. No es la selección final (esa se hace dentro del pliegue o por dominio, según el protocolo), sino una lista de qué candidatas entran al conjunto núcleo, cuáles salen y por qué.

In [10]:
def decidir(r):
    v = r["variable"]
    if v in {"recaudo"}:
        return "Fuera del núcleo", "Confundida con la época (directo hasta 2023Q3); pendiente de aclarar con la operación"
    if v in {"tipo_solicitud", "es_refinanciacion"}:
        return "Fuera del núcleo", "La etiqueta de refinanciación solo existe desde 2024Q2 (indicador de época)"
    if v in {"n_cuotas", "cuota_mensual"}:
        return "Fuera (redundante)", "Redundante con plazo / monto (Spearman ≥ 0,8)"
    if v == "monto":
        return "Núcleo (vigilar)", "Única variable sobre el IV de ruido, pero la señal no se repite con 60+/30+"
    if v == "emp_creditos_previos":
        return "Núcleo (vigilar)", "Señal consistente en las tres definiciones; evaluar dentro de cohorte (crece con el calendario)"
    if v in {"peor_mora_previa", "n_activos_t0"} and r["signo_observado"] != r["signo_esperado"]:
        return "Núcleo por dominio" if v == "peor_mora_previa" else "Candidata secundaria", \
            "Signo observado opuesto al esperado, IC incluye 0,5; se conserva por dominio, no por los datos"
    if v in {"n_activos_t0"}:
        return "Candidata secundaria", "Correlacionada con n_creditos_previos (0,75)"
    if v in {"cobra_seguro", "cobra_estudio"}:
        return "Fuera del núcleo", "Constante antes de 2025 (parámetro de producto introducido después)"
    if v in {"cobra_4x1000", "tiene_aval"}:
        return "Candidata secundaria", "Parámetro de producto que cambia con la política; revisar deriva (notebook 07)"
    if v == "tasa_mensual":
        return "Candidata secundaria", "Poca variación entre clientes; se fija por política"
    return "Núcleo", "Señal de dominio clara y disponible en t0 en todas las cohortes"


dec = biv[["variable", "iv", "auc", "signo_esperado", "signo_observado"]].copy()
dec[["propuesta", "razon"]] = dec.apply(lambda r: pd.Series(decidir(r)), axis=1)
dec = pd.concat([dec, pd.DataFrame([
    {"variable": "buró y PILA (todas)", "propuesta": "Fuera del modelo con datos propios",
     "razon": "0 % de datos en desarrollo; solo vía prior de Home Credit o en supervivencia con créditos recientes"},
    {"variable": "género", "propuesta": "Excluida", "razon": "Decisión 6 del Project"},
    {"variable": "huella digital", "propuesta": "Excluida", "razon": "Sin créditos maduros; la versión de la v1 era sintética"}])],
    ignore_index=True)
io_datos.guardar_resultado(dec.round(3), "fase2", "propuesta_variables_fase3")
dec.round(3)

,variable,iv,auc,signo_esperado,signo_observado,propuesta,razon
0,monto,0.626,0.666,?,+,Núcleo (vigilar),"Única variable sobre el IV de ruido, pero la s..."
1,cuota_mensual,0.451,0.678,?,+,Fuera (redundante),"Redundante con plazo / monto (Spearman ≥ 0,8)"
2,emp_creditos_previos,0.247,0.306,−,−,Núcleo (vigilar),Señal consistente en las tres definiciones; ev...
3,cierre_previo,0.213,NaN,NaN,NaN,Núcleo,Señal de dominio clara y disponible en t0 en t...
4,cobra_4x1000,0.121,0.591,?,+,Candidata secundaria,Parámetro de producto que cambia con la políti...
5,meses_desde_primer_credito,0.066,0.457,−,−,Núcleo,Señal de dominio clara y disponible en t0 en t...
6,plazo_meses,0.064,0.574,+,+,Núcleo,Señal de dominio clara y disponible en t0 en t...
7,edad_t0,0.063,0.426,−,−,Núcleo,Señal de dominio clara y disponible en t0 en t...
8,tasa_mensual,0.055,0.532,+,+,Candidata secundaria,Poca variación entre clientes; se fija por pol...
9,peor_mora_previa,0.032,0.436,+,−,Núcleo por dominio,"Signo observado opuesto al esperado, IC incluy..."


In [11]:
_ = io_datos.escribir_registro("06_eda_bivariado_woe_iv", {
    "poblacion": "libranza elegible < 2025", "creditos": len(dev), "eventos": int(dev["y"].sum()),
    "bootstrap": B, "iv_ruido_p95": iv_ruido_p95, "figuras": ["E06", "E07", "E08"]})

## Conclusiones

- Con 16 eventos, **ninguna variable tiene una asociación estimada con precisión**. Los IC del IV y del AUC univariado son muy anchos, y el IV de una variable de puro ruido ya es considerable.
- El recaudo casi no discrimina dentro de libranza en desarrollo (IV 0,01), y dentro de 2023 la relación se **invierte** (empresarial 6,8 % frente a mensual 1,5 %). La señal "recaudo mensual = más riesgo" de la v1 venía de mezclar productos y cohortes.
- La única variable que supera el IV esperado por ruido es el **monto**, pero su señal no se repite con 60+/12m ni con 30+/6m. La señal más **consistente** entre definiciones es la antigüedad del empleador (`emp_creditos_previos`), con la salvedad de que también crece con el calendario.
- La peor mora previa y los créditos activos en t0 no muestran el signo esperado (sus IC incluyen 0,5): no hay evidencia de señal con 16 eventos.
- La propuesta para la Fase 3 es un núcleo de variables de historia del cliente, términos del crédito (monto, plazo), edad y antigüedad del empleador, **todas disponibles en t0 en todas las cohortes**. El conjunto reducido es coherente con el límite de parámetros de la Fase 1.
- Las variables que son indicadores de época (recaudo, tipo de solicitud) quedan fuera del núcleo mientras no se aclare su significado con la operación.